# Karate Club dataset

This notebook shows the classic Zachary Karate Club network loaded into both
`NetworkXGraph` and `Neo4jGraph` using the shared loader in `cvcdocdb.exemples`.

Dataset notes:
- 34 members
- classic community-detection benchmark
- `club` attribute is used to build club nodes

In [ ]:
import importlib.util

# Comprovació de presència del paquet
package_to_check = 'cvcdocdb'
spec = importlib.util.find_spec(package_to_check)

if spec is None:
    print(f'⚠️ {package_to_check} no està instal·lat. Iniciant instal·lació...')
    %pip install -q --upgrade cvcdocdb
    print("✅ Instal·lació completada. L'estat del kernel PODRIA requerir un reinici.")
else:
    print(f'✅ {package_to_check} ja està present al sistema. Saltant instal·lació.')


In [7]:
import os

from cvcdocdb import Neo4jGraph, NetworkXGraph
from cvcdocdb.exemples import load_karate_club


def neo4j_config(default_target="DEV"):
    target = os.environ.get("NEO4J_TARGET", default_target).upper()
    prefix = f"NEO4J_{target}_"
    return {
        "target": target,
        "url": os.environ.get(f"{prefix}URL", os.environ.get("NEO4J_URL", "bolt://localhost:7687")),
        "user": os.environ.get(f"{prefix}USER", os.environ.get("NEO4J_USER", "neo4j")),
        "password": os.environ.get(f"{prefix}PASSWORD", os.environ.get("NEO4J_PASSWORD", "secret")),
        "database": os.environ.get(f"{prefix}DATABASE", os.environ.get("NEO4J_DATABASE", "neo4j")),
    }

## NetworkXGraph

In [8]:
nx_graph = NetworkXGraph()
nx_stats = load_karate_club(nx_graph)
print("NetworkX stats:", nx_stats)
print("NetworkX nodes:", len(nx_graph.get_nodes()))
print("NetworkX edges:", len(nx_graph.get_edges()))
nx_graph.close()

NetworkX stats: {'members': 34, 'clubs': 2, 'interacts': 78, 'member_of': 34}
NetworkX nodes: 36
NetworkX edges: 112


## Neo4jGraph

In [ ]:
cfg = neo4j_config()
print("Using Neo4j target:", cfg["target"])
neo4j_graph = Neo4jGraph(
    url=cfg["url"],
    user=cfg["user"],
    password=cfg["password"],
    database=cfg["database"],
)
try:
    neo4j_stats = load_karate_club(neo4j_graph)
    print("Neo4j stats:", neo4j_stats)
finally:
    neo4j_graph.close()
